# Using Alphafold components on Google Colab

---



In [2]:
#@title Install dependencies
#@markdown * Copied from the [Alphafold Colab notebook](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)

from IPython.utils import io
import os
import subprocess
import tqdm.notebook

TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

GIT_REPO = 'https://github.com/deepmind/alphafold'
DATA_REPO = 'https://github.com/sameerd/data_dump'
TMSCORE_BIN = "./data_dump/TMscore/TMscore"

force_reinstall_dependencies = False  #@param {type:"boolean"}

if (not os.path.exists("INSTALLED_DEPS")) or force_reinstall_dependencies:
  try:
    with tqdm.notebook.tqdm(total=100, bar_format=TQDM_BAR_FORMAT) as pbar:
      with io.capture_output() as captured:
        %shell rm -f INSTALLED_DEPS
        # Uninstall default Colab version of TF.
        %shell pip uninstall -y tensorflow
        pbar.update(6)
        %shell rm -rf alphafold
        %shell git clone --branch main {GIT_REPO} alphafold
        pbar.update(8)
        %shell pip3 install -r ./alphafold/requirements.txt
        pbar.update(46)
        # Run setup.py to install only AlphaFold.
        %shell pip3 install --no-dependencies ./alphafold
        pbar.update(28)
        # add optax in case we want to use an optimizer
        %shell pip3 install --no-dependencies optax
        pbar.update(4)
        %shell rm -rf data_dump
        %shell git clone --branch main {DATA_REPO} data_dump
        pbar.update(4)
        %shell (cd data_dump/TMscore; make)
        pbar.update(4)
        %shell touch INSTALLED_DEPS
  except subprocess.CalledProcessError:
    print(captured)
    raise


import jax
if jax.local_devices()[0].platform == 'tpu':
  raise RuntimeError('Colab TPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
elif jax.local_devices()[0].platform == 'cpu':
  #raise RuntimeError('Colab CPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
  print(f"{jax.local_devices()}")
else:
  print(f'Running with {jax.local_devices()[0].device_kind} GPU')


# Make sure all necessary environment variables are set.
import os
os.environ['TF_FORCE_UNIFIED_MEMORY'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '2.0'





  0%|          | 0/100 [elapsed: 00:00 remaining: ?]

[CpuDevice(id=0)]


In [3]:
from re import T
from typing import Any, Mapping, Optional, Union

import dataclasses
import gzip
import pathlib

import numpy as np

#@markdown ### Set global seed
SEED = 100 #@param 

import alphafold
import alphafold.common
from alphafold.common.protein import from_pdb_string
from alphafold.common.residue_constants import restypes

def convert_aatype_to_string(prot_aatype : np.ndarray):
  # convert amino acid indices in protein to a sequence string
  return "".join(restypes[i] for i in prot_aatype)

def read_pdb_gz(filename):
  with gzip.open(filename, "rt") as fh:
    return from_pdb_string(fh.read())

#@markdown ### Read in the starting sequence
data_dir = pathlib.Path("data_dump")
starting_filename = "starting.pdb.gz" #@param
s_prot = read_pdb_gz(data_dir/ starting_filename)
starting_seq = convert_aatype_to_string(s_prot.aatype)
num_res = len(starting_seq)

print(f"Seq : {starting_seq}\nNum Residues : {num_res}")
for f in dataclasses.fields(s_prot):
  print(f"{f.name:15s}{f.type},  {getattr(s_prot, f.name).shape}")
print()

training_directory = "training_pdbs" #@param
training_pdb_list = list((data_dir / training_directory).glob("*.pdb.gz"))
N_all = len(training_pdb_list)
print(f"Num structures in dataset : {N_all}")

validation_fraction = 0.1 #@param
N_val = int(N_all * validation_fraction) 
N_train = N_all - N_val

np.random.seed(SEED)
train_indices = np.random.choice(N_all, size=N_train, replace=False)
all_data_mask = np.zeros(N_all, dtype=bool)
all_data_mask[train_indices] = True
val_indices = np.where(~all_data_mask)[0]

assert(len(train_indices) == N_train)
assert(len(val_indices) == N_val)

print(f"Splitting dataset         : N_train: {N_train}, N_val={N_val}")


Seq : MQHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGERDAWWDDEGFSSSPFTKNAHHAGIVATSVTLGQLQREQGDKLVSKAAEYFGIACRVNDGLRTTRFVRLFSDALDAKPLTIGHDYEVEFLLATRRVYEPFEAPFNFAPHCDDVSYGRDTVNWPLKRSFPRQLGGFLTIQGADNDAGMVMWDNRPESRAALDEMHAEYRETGAIAALERAAKIMLKPQPGQLTLFQSKNLHAIERCTSTRRTMGLFLIHTEDGWRMFD
Num Residues : 273
atom_positions <class 'numpy.ndarray'>,  (273, 37, 3)
aatype         <class 'numpy.ndarray'>,  (273,)
atom_mask      <class 'numpy.ndarray'>,  (273, 37)
residue_index  <class 'numpy.ndarray'>,  (273,)
chain_index    <class 'numpy.ndarray'>,  (273,)
b_factors      <class 'numpy.ndarray'>,  (273, 37)

Num structures in dataset : 1000
Splitting dataset         : N_train: 900, N_val=100


## Model

In [4]:
from alphafold.model.common_modules import Linear

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk

In [5]:
def _f(x):
    m = Linear(num_output=2, num_input_dims=1, name="linear22")
    return m(x)
f = hk.transform(_f)
f

Transformed(init=<function without_state.<locals>.init_fn at 0x7fe362ded7a0>, apply=<function without_state.<locals>.apply_fn at 0x7fe362ded560>)

In [6]:
test_inp = s_prot.atom_positions.astype(jnp.float32)

In [7]:
rng_key = jax.random.PRNGKey(42)
params = f.init(x=test_inp, rng=rng_key)
params

FlatMapping({
  'linear22': FlatMapping({
                'weights': DeviceArray([[-0.34503725,  0.582498  ],
                                        [ 0.0910517 , -0.35958534],
                                        [ 0.68855035,  0.66172487]], dtype=float32),
                'bias': DeviceArray([0., 0.], dtype=float32),
              }),
})

In [8]:
s_prot.atom_positions.shape

(273, 37, 3)

In [9]:
output1 = f.apply(params, rng=rng_key, x=test_inp)
output1

DeviceArray([[[ -1.5173626 , -46.807423  ],
              [ -0.51049376, -45.875412  ],
              [ -0.74139994, -44.870083  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[ -0.21452728, -43.684746  ],
              [ -0.28924042, -42.55946   ],
              [  0.38171828, -42.358902  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[  0.0673494 , -41.861176  ],
              [  0.7287432 , -41.54246   ],
              [  1.6379772 , -40.37801   ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             ...,

             [[ -6.134639  , -31.984861  ],
              [ -6.269721  , -31.408993  ],
              [ -6.9840364 , -30.61801

In [10]:
from alphafold.model import config
model_name = ('model_1')
cfg = config.model_config(model_name)
cfg.data.eval.num_ensemble = 1

In [11]:
from alphafold.model.modules import dgram_from_positions
from alphafold.model.tf.data_transforms import pseudo_beta_fn

dgram_features = cfg.model.embeddings_and_evoformer.template.dgram_features
print(dgram_features)

# create pseudo beta for glycines # convert tensorflow array to numpy
pseudo_beta_pos = pseudo_beta_fn(s_prot.aatype, s_prot.atom_positions, 
                                 all_atom_masks = None).numpy() 
s_dgram = dgram_from_positions(pseudo_beta_pos, **dgram_features)
s_dgram.shape

{max_bin: 50.75, min_bin: 3.25, num_bins: 39}



(273, 273, 39)

In [12]:
from alphafold.common import residue_constants
from alphafold.model import quat_affine

n, ca, c = [residue_constants.atom_order[a] for a in ('N', 'CA', 'C')]
rot, trans = quat_affine.make_transform_from_reference(
        n_xyz=s_prot.atom_positions[:, n],
        ca_xyz=s_prot.atom_positions[:, ca],
        c_xyz=s_prot.atom_positions[:, c])
affines = quat_affine.QuatAffine(
        quaternion=quat_affine.rot_to_quat(rot, unstack_inputs=True),
        translation=trans,
        rotation=rot,
        unstack_inputs=True)
points = [jnp.expand_dims(x, axis=-2) for x in affines.translation]
affine_vec = affines.invert_point(points, extra_dims=1)

In [13]:
# make features
import alphafold.data.pipeline as pipeline
import alphafold.model.features as features

features_dict = pipeline.make_sequence_features(starting_seq, description="query", num_res=num_res)
del features_dict['between_segment_residues']
del features_dict['domain_name']
del features_dict['sequence']
features_dict

{'aatype': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=int32),
 'residue_index': array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131,

In [14]:
cfg.data.common.unsupervised_features

['aatype',
 'residue_index',
 'sequence',
 'msa',
 'domain_name',
 'num_alignments',
 'seq_length',
 'between_segment_residues',
 'deletion_matrix']

In [15]:
import logging
import alphafold.model.folding
import tensorflow.compat.v1 as tf

In [16]:
class RunTestModel:
  """ Container for Jax model. """

  def __init__(self, 
               config,
               params: Optional[Mapping[str, Mapping[str, np.ndarray]]] = None):
    self.config = config
    self.params = params

    def _forward_fn(batch):
      model = alphafold.model.folding.StructureModule(config=self.config.model, 
                  global_config=self.config.model.global_config,
                  compute_loss=True)
      return model(batch, is_training=True)
    self.apply = jax.jit(hk.transform(_forward_fn).apply)
    self.init = jax.jit(hk.transform(_forward_fn).init)

  def init_params(self, feat: features.FeatureDict, random_seed: int = 0):
    """Initializes the model parameters.
    If none were provided when this class was instantiated then the parameters
    are randomly initialized.
    Args:
      feat: A dictionary of NumPy feature arrays as output by
        RunModel.process_features.
      random_seed: A random seed to use to initialize the parameters if none
        were set when this class was initialized.
    """
    if not self.params:
      # Init params randomly.
      rng = jax.random.PRNGKey(random_seed)
      self.params = hk.data_structures.to_mutable_dict(
          self.init(rng, feat))
      logging.warning('Initialized parameters randomly')

In [17]:
model_runner = RunTestModel(cfg)


In [22]:
%shell {TMSCORE_BIN}


 Brief instruction for running TM-score program:
 (For detail: Zhang & Skolnick, Proteins, 2004 57:702-10)

 1. Run TM-score to compare 'model.pdb' and 'native.pdb':
     $ TMscore model.pdb native.pdb

 2. Run TM-score to compare two complex structures with multiple chains
     $ TMscore -c model.pdb native.pdb

 3. TM-score normalized with an assigned scale d0 e.g. 5 A:
     $ TMscore model.pdb native.pdb -d 5

 4. TM-score normalized by a specific length, e.g. 120 AA:
     $ TMscore model.pdb native.pdb -l 120

 5. TM-score with superposition output, e.g. 'TM_sup*':
     $ TMscore model.pdb native.pdb -o TM_sup
    View superposed CA-traces by RasMol or PyMOL:
     $ rasmol -script TM_sup
     $ pymol -d @TM_sup.pml
    View superposed atomic models by RasMol:
     $ rasmol -script TM_sup_atm
     $ pymol -d @TM_sup_atm.pml

 6. For full help message:
    $ TMscore -h

